# 01 - Data Understanding

This notebook explores the M5 dataset structure and relationships.

## Objectives:
- Load all datasets
- Understand column meanings
- Explore data relationships
- Get basic statistics

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.utils import get_data_summary, get_date_range_info

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('default')
sns.set_palette('husl')

## 1. Load Data

In [ ]:
loader = DataLoader(data_dir='../data/raw')
data = loader.load_all_data()

calendar_df = data['calendar']
sales_df = data['sales']
prices_df = data['prices']

print("Data loaded successfully!")
print(f"\nCalendar shape: {calendar_df.shape}")
print(f"Sales shape: {sales_df.shape}")
print(f"Prices shape: {prices_df.shape}")

## 2. Calendar Data

In [ ]:
print("Calendar Data Info:")
print("\nColumns:", list(calendar_df.columns))
print("\nFirst few rows:")
calendar_df.head(10)

In [ ]:
date_info = get_date_range_info(calendar_df, 'date')
print("Date Range Information:")
for key, value in date_info.items():
    print(f"  {key}: {value}")

In [ ]:
print("\nEvent Analysis:")
print(f"Days with events: {(calendar_df['event_name_1'] != 'NA').sum()}")
print(f"\nEvent types:")
print(calendar_df['event_type_1'].value_counts())

## 3. Sales Data

In [ ]:
print("Sales Data Info:")
print("\nID Columns:", [col for col in sales_df.columns if not col.startswith('d_')])
print("\nNumber of day columns:", len([col for col in sales_df.columns if col.startswith('d_')]))
print("\nFirst few rows (ID columns only):")
id_cols = [col for col in sales_df.columns if not col.startswith('d_')]
sales_df[id_cols].head()

In [ ]:
print("\nData Hierarchy:")
print(f"States: {sales_df['state_id'].nunique()} - {sales_df['state_id'].unique()}")
print(f"Stores: {sales_df['store_id'].nunique()} - {sales_df['store_id'].unique()}")
print(f"Categories: {sales_df['cat_id'].nunique()} - {sales_df['cat_id'].unique()}")
print(f"Departments: {sales_df['dept_id'].nunique()}")
print(f"Items: {sales_df['item_id'].nunique()}")
print(f"\nTotal time series: {len(sales_df)}")

## 4. Prices Data

In [ ]:
print("Prices Data Info:")
print("\nColumns:", list(prices_df.columns))
print("\nFirst few rows:")
prices_df.head()

In [ ]:
print("\nPrice Statistics:")
print(prices_df['sell_price'].describe())

plt.figure(figsize=(10, 5))
plt.hist(prices_df['sell_price'], bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.show()

## 5. Data Summary

In [ ]:
print("Overall Data Summary:")
for name, df in [('Calendar', calendar_df), ('Sales', sales_df), ('Prices', prices_df)]:
    summary = get_data_summary(df)
    print(f"\n{name}:")
    for key, value in summary.items():
        print(f"  {key}: {value}")

## 6. Data Relationships

In [ ]:
print("Data Relationships:")
print("\nCalendar-Sales relationship:")
print(f"  Calendar has {calendar_df.shape[0]} days")
print(f"  Sales has {len([col for col in sales_df.columns if col.startswith('d_')])} day columns")
print(f"  Match: {calendar_df.shape[0] == len([col for col in sales_df.columns if col.startswith('d_')])}")

print("\nPrices-Sales relationship:")
sample_item = sales_df['item_id'].iloc[0]
sample_store = sales_df['store_id'].iloc[0]
price_records = prices_df[(prices_df['item_id'] == sample_item) & 
                          (prices_df['store_id'] == sample_store)]
print(f"  Sample item {sample_item} at store {sample_store} has {len(price_records)} price records")

## 7. Key Findings

- **Calendar**: Contains date information, events, and SNAP program flags
- **Sales**: Wide format with one row per item-store combination
- **Prices**: Item prices per store and week
- **Relationships**: Data can be merged using date (d_*), item_id, store_id, and wm_yr_wk